# Pipeline entregable FAT2019

Este notebook es la version preparada para reconstruir la entrega final en un entorno seguro. No se ejecuta durante la reorganizacion local.

Modelo final oficial:

```text
0.25  * separable_headsep_e100_seed42
+ 0.375 * globalmel_sep_temporal_e100_seed42
+ 0.375 * sep_temporal_f1024_e100_seed42
```

Kaggle private LB verificado: `0.67126`.


## 0. Configuracion

`RUN_HEAVY_STEPS=False` usa los CSV ya generados. `RUN_HEAVY_STEPS=True` reconstruye caches log-mel y reentrena las tres ramas a 100 epocas.


In [1]:
from __future__ import annotations

import hashlib
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

RUN_HEAVY_STEPS = True
RUN_OPTIONAL_KAGGLE_QUERY = False

ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'sample_submission.csv').exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('No pude encontrar la raiz del repo')
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data'
INVESTIGATION = ROOT / 'investigation'
SCRIPTS = INVESTIGATION / 'scripts'
DELIVERABLE = ROOT / '100. Entregable'
OUTPUTS = DELIVERABLE / 'outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)
VENV_PYTHON = ROOT / '.venv' / 'bin' / 'python'
PYTHON = VENV_PYTHON if VENV_PYTHON.exists() else Path(sys.executable)
print('subprocess_python', PYTHON)

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def run_command(args: list[str | Path], *, heavy: bool = False) -> None:
    rendered = ' '.join(str(arg) for arg in args)
    print(f'$ {rendered}')
    if heavy and not RUN_HEAVY_STEPS:
        print('SKIP: RUN_HEAVY_STEPS=False')
        return
    subprocess.run([str(arg) for arg in args], check=True, cwd=ROOT)


subprocess_python /home/santig14/fing/taa/2_TallerAA/.venv/bin/python


## 1. Datos y formato esperado

La competencia usa `sample_submission.csv` con `fname` y 80 clases. La salida final debe conservar exactamente esas filas y columnas.


In [2]:
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
label_columns = list(sample_submission.columns[1:])
{'test_rows': len(sample_submission), 'num_classes': len(label_columns)}


{'test_rows': 3361, 'num_classes': 80}

## 2. Preprocesamiento comun

El flujo de audio usado por las ramas es:

```text
.wav -> mono -> 44.1 kHz -> MelSpectrogram -> log/dB -> crop/pad -> normalizacion -> tensor 128 x T
```

Se preparan tres representaciones: `128 x 512` por clip, `128 x 512` con normalizacion global mel, y `128 x 1024` por clip.


In [3]:
run_command([PYTHON, SCRIPTS / 'build_logmel_image_cache.py', '--data-dir', DATA_DIR, '--n-mels', '128', '--frames', '512', '--normalization', 'clip'], heavy=True)
run_command([PYTHON, SCRIPTS / 'build_logmel_image_cache.py', '--data-dir', DATA_DIR, '--n-mels', '128', '--frames', '512', '--normalization', 'global-mel', '--cache-tag', 'globalmel'], heavy=True)
run_command([PYTHON, SCRIPTS / 'build_logmel_image_cache.py', '--data-dir', DATA_DIR, '--n-mels', '128', '--frames', '1024', '--normalization', 'clip'], heavy=True)


$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --n-mels 128 --frames 512 --normalization clip


skip existing /home/santig14/fing/taa/2_TallerAA/data/curated_logmel_image_m128_f512.npz
skip existing /home/santig14/fing/taa/2_TallerAA/data/test_logmel_image_m128_f512.npz
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --n-mels 128 --frames 512 --normalization global-mel --cache-tag globalmel


global stats: 250/4964


global stats: 500/4964


global stats: 750/4964


...

global stats: 4500/4964


global stats: 4750/4964


skip existing /home/santig14/fing/taa/2_TallerAA/data/curated_logmel_image_m128_f512_globalmel.npz
skip existing /home/santig14/fing/taa/2_TallerAA/data/test_logmel_image_m128_f512_globalmel.npz


$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --n-mels 128 --frames 1024 --normalization clip


skip existing /home/santig14/fing/taa/2_TallerAA/data/curated_logmel_image_m128_f1024.npz
skip existing /home/santig14/fing/taa/2_TallerAA/data/test_logmel_image_m128_f1024.npz


## 3. Entrenamiento de ramas e100

Las tres ramas comparten log-mel, pero difieren en arquitectura, normalizacion y longitud temporal.


### Regularizacion usada por las ramas finales

La configuracion final usa regularizacion en tres niveles:

- dentro del modelo: `head_dropout=0.3` en las tres ramas y `BatchNorm` en los bloques convolucionales;
- durante entrenamiento: augmentation tipo SpecAugment sobre train, con corrimiento temporal, mascara temporal y mascara en frecuencia;
- en el optimizador: `AdamW` con `weight_decay=1e-4` para `globalmel_sep_temporal` y `sep_temporal_f1024`.

`separable_headsep` usa `Adam`, y en `train_logmel_cnn.py` ese optimizador no recibe `weight_decay`, por lo que su regularizacion efectiva viene de dropout, BatchNorm y augmentation. En la corrida final quedan apagados `block_dropout`, `early_stopping`, time reverse y contraste aleatorio.


In [4]:
training_commands = {
    'separable_headsep_e100_seed42': [
        PYTHON, SCRIPTS / 'train_logmel_cnn.py', '--data-dir', DATA_DIR,
        '--models-dir', OUTPUTS / 'separable_headsep_e100' / 'models',
        '--submissions-dir', OUTPUTS / 'separable_headsep_e100' / 'submissions',
        '--experiments-dir', OUTPUTS / 'separable_headsep_e100' / 'experiments',
        '--n-mels', '128', '--frames', '512', '--architecture', 'separable_residual',
        '--activation', 'relu', '--initializer', 'he_normal', '--head-hidden', '256',
        '--head-dropout', '0.3', '--optimizer', 'adam', '--scheduler', 'multistep',
        '--lr-milestones', '27,36,43,49,52', '--epochs', '100', '--batch-size', '24',
        '--seed', '42', '--full-train',
    ],
    'globalmel_sep_temporal_e100_seed42': [
        PYTHON, SCRIPTS / 'train_logmel_cnn.py', '--data-dir', DATA_DIR,
        '--models-dir', OUTPUTS / 'globalmel_e100' / 'models',
        '--submissions-dir', OUTPUTS / 'globalmel_e100' / 'submissions',
        '--experiments-dir', OUTPUTS / 'globalmel_e100' / 'experiments',
        '--n-mels', '128', '--frames', '512', '--cache-tag', 'globalmel',
        '--architecture', 'separable_temporal_bigru', '--activation', 'silu',
        '--initializer', 'he_normal', '--head-dropout', '0.3', '--optimizer', 'adamw',
        '--scheduler', 'multistep', '--lr-milestones', '25,39', '--epochs', '100',
        '--batch-size', '24', '--seed', '42', '--full-train',
    ],
    'sep_temporal_f1024_e100_seed42': [
        PYTHON, SCRIPTS / 'train_logmel_cnn.py', '--data-dir', DATA_DIR,
        '--models-dir', OUTPUTS / 'f1024_e100' / 'models',
        '--submissions-dir', OUTPUTS / 'f1024_e100' / 'submissions',
        '--experiments-dir', OUTPUTS / 'f1024_e100' / 'experiments',
        '--n-mels', '128', '--frames', '1024', '--architecture', 'separable_temporal_bigru',
        '--activation', 'silu', '--initializer', 'he_normal', '--head-dropout', '0.3',
        '--optimizer', 'adamw', '--scheduler', 'multistep', '--lr-milestones', '19,25',
        '--epochs', '100', '--batch-size', '12', '--seed', '42', '--full-train',
    ],
}
for name, args in training_commands.items():
    print(name)
    run_command(args, heavy=True)


separable_headsep_e100_seed42
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/train_logmel_cnn.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --models-dir /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/separable_headsep_e100/models --submissions-dir /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/separable_headsep_e100/submissions --experiments-dir /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/separable_headsep_e100/experiments --n-mels 128 --frames 512 --architecture separable_residual --activation relu --initializer he_normal --head-hidden 256 --head-dropout 0.3 --optimizer adam --scheduler multistep --lr-milestones 27,36,43,49,52 --epochs 100 --batch-size 24 --seed 42 --full-train


device=cuda


epoch 1: loss=0.57822 lr=0.00100000


epoch 2: loss=0.44427 lr=0.00100000


epoch 3: loss=0.38244 lr=0.00100000


...

epoch 97: loss=0.01745 lr=0.00025000


epoch 98: loss=0.01547 lr=0.00025000


epoch 99: loss=0.01749 lr=0.00025000


epoch 100: loss=0.01740 lr=0.00025000


full_train_epoch=100
wrote /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/f1024_e100/submissions/small_logmel_cnn.csv


## 4. Seleccion de predicciones

En modo seguro se usan los CSV ya generados por `parallel100_20260702`. En modo pesado se usan las salidas recien entrenadas.


In [5]:
if RUN_HEAVY_STEPS:
    component_submissions = {
        'headsep': OUTPUTS / 'separable_headsep_e100' / 'submissions' / 'small_logmel_cnn.csv',
        'globalmel': OUTPUTS / 'globalmel_e100' / 'submissions' / 'small_logmel_cnn.csv',
        'f1024': OUTPUTS / 'f1024_e100' / 'submissions' / 'small_logmel_cnn.csv',
    }
else:
    component_submissions = {
        'headsep': ROOT / 'investigation/submissions/parallel100_20260702_separable_headsep_e100_seed42/small_logmel_cnn.csv',
        'globalmel': ROOT / 'investigation/submissions/parallel100_20260702_globalmel_sep_temporal_e100_seed42/small_logmel_cnn.csv',
        'f1024': ROOT / 'investigation/submissions/parallel100_20260702_sep_temporal_f1024_e100_seed42/small_logmel_cnn.csv',
    }
for name, path in component_submissions.items():
    print(name, path, path.exists())
    if not path.exists():
        raise FileNotFoundError(path)


headsep /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/separable_headsep_e100/submissions/small_logmel_cnn.csv True
globalmel /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/globalmel_e100/submissions/small_logmel_cnn.csv True
f1024 /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/f1024_e100/submissions/small_logmel_cnn.csv True


## 5. Blend final

El blend final promedia probabilidades clase a clase con pesos `0.25`, `0.375` y `0.375`.


In [6]:
final_submission_path = DELIVERABLE / 'submission.csv'
blend_args = [
    PYTHON, SCRIPTS / 'blend_submissions.py',
    '--input', component_submissions['headsep'], '--weight', '0.25',
    '--input', component_submissions['globalmel'], '--weight', '0.375',
    '--input', component_submissions['f1024'], '--weight', '0.375',
    '--output', final_submission_path,
]
run_command(blend_args)
print('final', final_submission_path)
print('sha256', sha256(final_submission_path))


$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/blend_submissions.py --input /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/separable_headsep_e100/submissions/small_logmel_cnn.csv --weight 0.25 --input /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/globalmel_e100/submissions/small_logmel_cnn.csv --weight 0.375 --input /home/santig14/fing/taa/2_TallerAA/100. Entregable/outputs/f1024_e100/submissions/small_logmel_cnn.csv --weight 0.375 --output /home/santig14/fing/taa/2_TallerAA/100. Entregable/submission.csv


wrote /home/santig14/fing/taa/2_TallerAA/100. Entregable/submission.csv from 3 submissions
final /home/santig14/fing/taa/2_TallerAA/100. Entregable/submission.csv
sha256 8e6e914924764fbd3cac5c3802b6861ebc25dcc1a84d41a4d769f2b0d306d6d2


## 6. Validacion

La submission debe coincidir con `sample_submission.csv` en filas y columnas, y sus probabilidades deben estar en `[0, 1]`.


In [7]:
final_df = pd.read_csv(final_submission_path)
assert list(final_df.columns) == list(sample_submission.columns)
assert final_df['fname'].equals(sample_submission['fname'])
assert final_df[label_columns].ge(0.0).all().all()
assert final_df[label_columns].le(1.0).all().all()
{'rows': len(final_df), 'columns': len(final_df.columns), 'sha256': sha256(final_submission_path)}


{'rows': 3361,
 'columns': 81,
 'sha256': '8e6e914924764fbd3cac5c3802b6861ebc25dcc1a84d41a4d769f2b0d306d6d2'}

## 7. Resultado registrado

Resultado Kaggle private para este CSV: `0.67126`.

El `0.67674` con bagging queda como mejor experimento, no como entrega principal.
